<a href="https://colab.research.google.com/github/fphsFischmeister/ILAE_NeuroimagingSchool/blob/master/notebooks/01_preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Google Colab"/>  </a>

# Welcome to the interactive ILAE workshop on task-based activation detection.

Author: Florian Ph.S Fischmeister, Marc Berger, Radeshyam Stepponat

---
- Part 1: Preprocessing fMRI data with fMRIPrep [Jupyter Notebook](https://colab.research.google.com/github/fphsFischmeister/ILAE_NeuroimagingSchool/blob/master/notebooks/01_preprocessing.ipynb)
- Part 2: First Level of a simple motor task [Jupyter Notebook](https://colab.research.google.com/github/fphsFischmeister/ILAE_NeuroimagingSchool/blob/master/notebooks/02_basic_motor_task.ipynb)
- Part 3: A Home-Town-Walking language paradigm [Jupyter Notebook](https://colab.research.google.com/github/fphsFischmeister/ILAE_NeuroimagingSchool/blob/master/notebooks/03_hometown_task.ipynb)
- Part 4: Phrases, a language paradigm [Jupyter Notebook](https://colab.research.google.com/github/fphsFischmeister/ILAE_NeuroimagingSchool/blob/master/notebooks/04_phases_task.ipynb)
- Part 5: All language tasks [Jupyter Notebook](https://colab.research.google.com/github/fphsFischmeister/ILAE_NeuroimagingSchool/blob/master/notebooks/05_all_language_task.ipynb)
- Part 6: Functional Connectivity [Jupyter Notebook](https://colab.research.google.com/github/fphsFischmeister/ILAE_NeuroimagingSchool/blob/master/notebooks/06_connectivity.ipynb)



In [ ]:
# get some data for presentation
!rm -rf ILAE_NeuroimagingSchool
!git clone https://github.com/fphsFischmeister/ILAE_NeuroimagingSchool.git

# Part 1: Preprocessing fMRI data with fMRIPrep

In this notebook, we introduce the preprocessing step required before task-based and functional-connectivity analyses of fMRI data in patients with epilepsy. The aim of preprocessing is to transform raw MRI and fMRI data into a standardized, quality-controlled set of derivative files that can be used in later statistical analyses.

We use fMRIPrep, a widely used and robust preprocessing pipeline for task-based and resting-state fMRI. Preprocessing is a critical step in fMRI analysis, converting raw fMRI data into a form suitable for later statistical analysis. fMRIPrep is designed as an analysis-agnostic workflow: it performs the main image-preparation steps, including coregistration, normalization, susceptibility distortion correction when appropriate, tissue segmentation, skull stripping, motion estimation, and extraction of confound measures, while leaving analysis-specific decisions such as smoothing, nuisance regression, temporal filtering, task modelling, and connectivity estimation to later stages. As a NiPreps application, fMRIPrep combines tools from established neuroimaging software packages, including FSL, ANTs, FreeSurfer, and AFNI, and provides BIDS-compatible outputs that can be used for a wide range of downstream analyses, including task-based fMRI, resting-state connectivity, graph-theory measures, and surface- or volume-based statistics. It also generates visual quality-control reports that help users inspect preprocessing results and identify potential outliers or processing failures.

**Note:** For epilepsy neuroimaging, careful preprocessing is particularly important. Patients may move during scanning, structural abnormalities may affect anatomical normalization, and prior surgery or focal lesions can complicate registration. Therefore, the output of fMRIPrep should not be treated as automatically “correct”; each subject’s report should be visually inspected before the data are used for task activation maps or connectivity analyses. Thus, we will mainly explore examples of the outputs it generates.


### Citations

- Esteban, O., Markiewicz, C.J., Blair, R.W. et al. fMRIPrep: a robust preprocessing pipeline for functional MRI. Nat Methods 16, 111–116 (2019). https://doi.org/10.1038/s41592-018-0235-4
- for a full Documentation see [https://fmriprep.org/en/stable/](https://fmriprep.org/en/stable/)


## fMRIprep workflow

![fMRIPrep workflow](./ILAE_NeuroimagingSchool/notebooks/fmriprep-21.0.0.svg)


## Preparing the data in BIDS format

fMRIPrep expects the input dataset to follow the [Brain Imaging Data Structure, or BIDS](https://bids.neuroimaging.io/index.html). At minimum, the dataset should include a T1-weighted anatomical image and, unless anatomical-only processing is requested, one or more BOLD fMRI runs. fMRIPrep’s documentation recommends validating the dataset with the BIDS Validator before running preprocessing.

In this notebook, the expected folder structure is:

````
sub-ILAEDemo001
    ├── dataset_description.json
    └── ses-01
        ├── anat
        │   ├── sub-ILAEDemo001_ses-01_run-01_T1w.json
        │   └── sub-ILAEDemo001_ses-01_run-01_T1w.nii.gz
        ├── func
        │   ├── sub-ILAEDemo001_ses-01_task-HomeTownWalking_run-01_bold.json
        │   ├── sub-ILAEDemo001_ses-01_task-HomeTownWalking_run-01_bold.nii.gz
        │   ├── sub-ILAEDemo001_ses-01_task-HomeTownWalking_run-01_events.tsv
        │   ├── sub-ILAEDemo001_ses-01_task-MotorHandright_run-01_bold.json
        │   ├── sub-ILAEDemo001_ses-01_task-MotorHandright_run-01_bold.nii.gz
        │   ├── sub-ILAEDemo001_ses-01_task-MotorHandright_run-01_events.tsv
        │   ├── sub-ILAEDemo001_ses-01_task-ObjectNaming_run-01_bold.json
        │   ├── sub-ILAEDemo001_ses-01_task-ObjectNaming_run-01_bold.nii.gz
        │   ├── sub-ILAEDemo001_ses-01_task-ObjectNaming_run-01_events.tsv
        │   ├── sub-ILAEDemo001_ses-01_task-Phrases_run-01_bold.json
        │   ├── sub-ILAEDemo001_ses-01_task-Phrases_run-01_bold.nii.gz
        │   ├── sub-ILAEDemo001_ses-01_task-Phrases_run-01_events.tsv
        │   ├── sub-ILAEDemo001_ses-01_task-VerbGeneration_run-01_bold.json
        │   ├── sub-ILAEDemo001_ses-01_task-VerbGeneration_run-01_bold.nii.gz
        │   └── sub-ILAEDemo001_ses-01_task-VerbGeneration_run-01_events.tsv
        └── sub-ILAEDemo001_ses-01_scans.tsv
````

## Running fMRIprep

```bash
fmriprep /data/bids /data/derivatives participant \
  --participant-label ILAEDemo001 \
  --output-spaces T1w \
  --fs-license-file /content/license.txt \
  --skull-strip-t1w  force\
  --fs-no-reconall \
  --skip-bids-validation \
  -w /content/work
```
For teaching purposes, we inspect precomputed fMRIPrep outputs. Full fMRIPrep processing can be computationally demanding. The fMRIPrep FAQ recommends processing one subject per container instance and gives approximate resource guidance of about 8 GB of memory for a single-subject preprocessing run without FreeSurfer surface processing.


## Visual quality check

The full output of the above dataset can be found at: https://dinlab.roentgen.meduniwien.ac.at/ILAE_NeuroimagingSchool/fmriprep/

## fMRIprep Results

### T1 and brain mask
For the anatomical image, fMRIPrep estimates a brain mask, performs tissue segmentation, and computes spatial normalization to template space. If FreeSurfer processing is enabled, cortical surface reconstruction can also be performed. For patients with focal lesions, resections, or other structural abnormalities, normalization should be inspected carefully.

Template T1-weighted image (if several T1w images were found), with contours delineating the detected brain mask and brain tissue segmentations:

In [ ]:
from IPython.core.display import SVG
SVG(filename='./ILAE_NeuroimagingSchool/fMRIprep_output/figures/sub-ILAEDemo001_ses-01_run-01_dseg.svg')

### Alignment of functional and anatomical MRI data (coregistration EPI-space to T1-space)
Alignment of the BOLD reference image to the anatomical (T1-weighted) image. The BOLD reference has been contrast enhanced for improved anatomical fidelity. The anatomical image has been resampled into BOLD reference space, as well as the anatomical white matter mask, which appears as a red contour.


In [ ]:
from IPython.core.display import SVG
SVG(filename='./ILAE_NeuroimagingSchool/fMRIprep_output/figures/sub-ILAEDemo001_ses-01_task-HomeTownWalking_run-01_desc-rois_bold.svg')


## Brain mask and (anatomical/temporal) CompCor ROIs
Brain mask calculated on the BOLD signal (red contour), along with the regions of interest (ROIs) used for the estimation of physiological and movement confounding components that can be then used as nuisance regressors in analysis.
The anatomical CompCor ROI (magenta contour) is a mask combining CSF and WM (white-matter), where voxels containing a minimal partial volume of GM have been removed.
The temporal CompCor ROI (blue contour) contains the top 2% most variable voxels within the brain mask.
The brain edge (or crown) ROI (green contour) picks signals outside but close to the brain, which are decomposed into 24 principal components.

In [ ]:
from IPython.core.display import SVG
SVG(filename='./ILAE_NeuroimagingSchool/fMRIprep_output/figures/sub-ILAEDemo001_ses-01_task-HomeTownWalking_run-01_desc-rois_bold.svg')

### Bold Summary
Summary statistics are plotted, which may reveal trends or artifacts in the BOLD data. Global signals calculated within the whole-brain (GS), within the white-matter (WM) and within cerebro-spinal fluid (CSF) show the mean BOLD signal in their corresponding masks. DVARS and FD show the standardized DVARS (the derivative of RMS variance over voxels) and framewise-displacement (quantification of the estimated bulk-head motion) measures for each time point.

A carpet plot shows the time series for all voxels within the brain mask. Voxels are grouped into cortical (dark/light blue), and subcortical (orange) gray matter, cerebellum (green) and white matter and CSF (red), indicated by the color map on the left-hand side.



In [ ]:
from IPython.core.display import SVG
SVG(filename='./ILAE_NeuroimagingSchool/fMRIprep_output/figures//sub-ILAEDemo001_ses-01_task-HomeTownWalking_run-01_desc-carpetplot_bold.svg')

In [ ]:
## Inspect the T1w image

In [ ]:
# installing NiiVue
!pip install ipyniivue

In [ ]:

from ipyniivue import NiiVue, ShowRender, SliceType

nv = NiiVue()
nv.load_volumes([{'path': './ILAE_NeuroimagingSchool/dataset/anat/sub-ILAEDemo001_ses-01_run-01_desc-preproc_T1w.nii.gz'}])
nv